In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from token import tok_name

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import os
embedding = HuggingFaceEmbeddings(
    # model_name="google/embeddinggemma-300m",
    # model_kwargs={"token": os.getenv("HFT")}
)
def load_pdf_into_vectorstore(path: str):
    document_loader = PyPDFLoader(path)
    documents = document_loader.load()
    document_chunks = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
    ).split_documents(documents)
    # embedding = HuggingFaceEmbeddings(model_name="google/embeddinggemma-300m")
    chroma_db = Chroma.from_documents(documents=document_chunks, embedding=embedding, persist_directory="./chroma-db")
    chroma_db.persist()
    print(len(document_chunks))
    print(len(documents))
    print("Uploaded document to vector store")
    return chroma_db


In [ ]:
# db = load_pdf_into_vectorstore("Griffiths - Introduction to Quantum Mechanics 3rd ed 2018.pdf")

vector_store = Chroma(persist_directory="./chroma-db", embedding_function=embedding)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={'k':10})

docs = retriever.invoke("What is Quantum mechanics in 4 bullet points ?")


In [ ]:
retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 15})


In [ ]:

llm = ChatOpenAI(model="gpt-5-nano")

prompt = """
    Retrieve the answers only from the context. if the context doesn't have anything related to the question. Answer "I don't know"

    {input}

    Context:
        {context}
"""

template = ChatPromptTemplate.from_messages([("human", prompt)])

chain = RunnableParallel({ "input": RunnablePassthrough(), "context": retriever}) | template | llm | StrOutputParser()

chain.invoke("who is bohr ?")